In [19]:
import yaml
import numpy as np
from matplotlib import pyplot as plt

# --- Custom Utility Module ---
from flux_history import FluxHistory
from mineral_utils import Paleodetector

In [20]:
MINERAL_LIBRARY = yaml.safe_load(open('Data/mineral_library.yaml', 'r'))

In [21]:
mineral_name =  "Quartz"

total_age_kyr = 300.

overburden_history_up = {
    "overburden_discrete": 30.,
    "density_discrete_g_cm3": 2.6,
    "time_discrete_kyr": 150.,
}

overburden_history_down = {
    "initial_depth": 30.,
    "initial_density_g_cm3": 2.6,
}

flux_history = FluxHistory()

In [22]:
mineral_config = MINERAL_LIBRARY.get(mineral_name)

mineral_up = Paleodetector(mineral_config, total_age_kyr=total_age_kyr, flux_history=flux_history, overburden_history=overburden_history_up)
mineral_down = Paleodetector(mineral_config, total_age_kyr=total_age_kyr, flux_history=flux_history, overburden_history=overburden_history_down)

Interpolating overburden history...
Created paleo-detector Quartz, species available for computation:
  neutron    energy from 0.001 to 171.9 GeV in 90 steps
  secondary_neutron neutrons produced by mu-, mu+, neutron above the sample (StdRock runs), recoils from the neutron runs
Interpolating overburden history...
Created paleo-detector Quartz, species available for computation:
  neutron    energy from 0.001 to 171.9 GeV in 90 steps
  secondary_neutron neutrons produced by mu-, mu+, neutron above the sample (StdRock runs), recoils from the neutron runs


In [23]:
x_bins, total_fission_tracks = mineral_up.integrate_fission_spectrum()
#x_bins, total_neutron_tracks = mineral_up.integrate_background_neutron_spectrum()

Calculating spontaneous fission background...


In [24]:
x_bins, tracks_up = mineral_up.integrate_all_particles()

  0%|                                                                                            | 0/2 [01:24<?, ?it/s]


FileNotFoundError: Data/Geant4_data/Quartz/Range/DEDX_Z4_A4.txt not found.

In [ ]:
x_bins, tracks_down = mineral_down.integrate_all_particles()

In [ ]:
x_mids = x_bins[:-1] + np.diff(x_bins) / 2

fig, ax = plt.subplots(figsize=[7, 5])

species = "total"

ax.step(x_mids/1000., total_fission_tracks, where='mid', label="Fission Tracks", color='blue')
ax.step(x_mids/1000., total_neutron_tracks, where='mid', label="Neutron Tracks", color='orange')

ax.step(x_mids/1000., tracks_up[species], where='mid', label="Tracks with surface exposure", color='green')
ax.step(x_mids/1000., tracks_down[species], where='mid', label="Tracks with no surface exposure", color='red')

ax.set_yscale("log")
ax.set_xlabel(r"Track Length ($\mu$m)")
ax.set_ylabel(r"N")
ax.set_title(f"Track number spectrum in 0.5 mg of {mineral_name}")
ax.set_ylim(0.05, 1e4)
ax.set_xlim(0., 30.)
ax.legend(fontsize='small', loc='upper right')
plt.show()